# Phase 3 — Ablation (A1–A4) and Hate Speech Benchmark

**Goal:** Quantify the contribution of peerness/repetition features (A2/A3/A4 vs A1) and benchmark text-only performance on the Automated Hate Speech dataset.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find F21MP project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.phase3_ablation_benchmark.evaluation.ablation import run_ablation, run_hate_speech_benchmark

## 1) Run ablation A1–A4 on Towards Comprehensive

Experiments:
- **A1**: text only
- **A2**: text + peerness
- **A3**: text + repetition
- **A4**: text + peerness + repetition (full)

In [2]:
ablation_out = run_ablation(save_outputs=True)
ablation_summary = ablation_out['summary']
ablation_summary

,experiment,accuracy,precision,recall,f1,roc_auc
0,A1_text_only,0.965992,0.000000,0.00,0.000000,0.509497
1,A2_text_plus_peerness,0.991093,0.540541,0.80,0.645161,0.995419
2,A3_text_plus_repetition,0.944130,0.025210,0.12,0.041667,0.729472
3,A4_full,0.999595,0.961538,1.00,0.980392,0.999984


## 2) Plot ablation F1

In [3]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(ablation_summary['experiment'], ablation_summary['f1'], color=['#1f77b4','#2ca02c','#ff7f0e','#9467bd'])
ax.set_title('Phase 3 Ablation (A1–A4): F1')
ax.set_ylabel('F1')
ax.set_xlabel('Experiment')
for i, v in enumerate(ablation_summary['f1']):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylim(0, min(1.05, max(ablation_summary['f1']) + 0.12))
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

/var/folders/mh/fg64trg179z805gx_j1k5b280000gn/T/ipykernel_36529/1331737698.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3) Run Automated Hate Speech benchmark (3-class)

Compares:
- TF-IDF + Logistic Regression
- TF-IDF + XGBoost

In [4]:
hs_out = run_hate_speech_benchmark(save_outputs=True)
hs_summary = hs_out['summary']
hs_summary

/Users/pharshitha/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,n_samples
0,tfidf_logreg,0.892677,0.807038,0.627424,0.656370,0.874876,4957
1,tfidf_xgboost,0.910026,0.775098,0.716196,0.724195,0.901194,4957


## 4) Phase 3 findings (summary)

### Ablation results (Towards Comprehensive)

| Experiment | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|
| **A1 text only** | 0.000 | 0.000 | 0.000 | 0.509 |
| **A2 text + peerness** | 0.541 | 0.800 | 0.645 | 0.995 |
| **A3 text + repetition** | 0.025 | 0.120 | 0.042 | 0.729 |
| **A4 full (text + peerness + repetition)** | 0.962 | 1.000 | 0.980 | 1.000 |

Interpretation:
- **A2 >> A1**: peerness contributes a large gain over text-only.
- **A3 > A1 but weak**: repetition features alone help slightly, but not enough.
- **A4 best overall**: combining peerness + repetition with text gives the strongest performance.

### Automated Hate Speech benchmark (3-class)

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|---:|
| TF-IDF + Logistic Regression | 0.893 | 0.807 | 0.627 | 0.656 | 0.875 |
| TF-IDF + XGBoost | **0.910** | 0.775 | **0.716** | **0.724** | **0.901** |

Interpretation:
- XGBoost achieves better macro-F1 and weighted-F1 than logistic regression.
- Macro-F1 remains below in-domain A4, consistent with cross-domain difficulty seen in Phase 2.

## 5) Saved outputs

- `outputs/results/phase3_ablation_a1_a4.json`
- `outputs/results/phase3_ablation_a1_a4_summary.csv`
- `outputs/results/phase3_hate_speech_benchmark.json`
- `outputs/results/phase3_hate_speech_benchmark_summary.csv`
- `outputs/media/phase3_ablation_f1.png`

## 6) Cross-reference: latest Phase 2 improvement path

To keep reporting consistent across phases, note that Phase 2 now includes an implemented "fastest to strongest" enhancement path on the Twitter subset:

- Small domain-adaptive fine-tune (text-only), threshold selected on calibration split
- Embedding-style extension (TF-IDF + SVD + Logistic Regression)

Latest reported test metrics from Phase 2:
- Domain-adaptive fine-tune: **F1 = 0.6334**, **ROC-AUC = 0.8028** (threshold = 0.40)
- Embedding extension: **F1 = 0.6441**, **ROC-AUC = 0.8172** (threshold = 0.30)

For full cells/results, see: `notebooks/phase2_validation/02_validation.ipynb`.
